In [2]:
from transformers import AutoTokenizer, AutoProcessor
hf_model = "bytedance-research/ChatTS-8B"
tokenizer = AutoTokenizer.from_pretrained(hf_model, trust_remote_code=True)
processor = AutoProcessor.from_pretrained(hf_model, trust_remote_code=True, tokenizer=tokenizer)
from model_factory.chatts import ChatTSEncoder

model = ChatTSEncoder()

/scratch/leo/envs/llm_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-04 15:36:23.598386: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-04 15:36:23.637632: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-04 15:36:24.661955: I tensorflow/core/util/port.cc:153] oneDNN custom operations are 

In [3]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from util.dataset import ChatTSDataset
from sklearn.metrics import f1_score, balanced_accuracy_score
from tqdm import tqdm
import random
import numpy as np

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def run_chatts_linear_probe(
    dataset_path,
    model,
    processor,
    batch_size=16,
    probe_batch_size=1024,
    lr=1e-4,
    weight_decay=5e-2,
    epochs=50,
    num_workers=4,
    device=None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1. Prepare Datasets
    dataset_train = ChatTSDataset(data_dir=dataset_path, is_zs=False, split="train")
    dataset_val = ChatTSDataset(data_dir=dataset_path, is_zs=False, split="test")

    def ChatTScollator(processor, batch):
        ts_list = []
        formatted_prompts = []
        for item in batch:
            # item["sensor"] is usually a list of arrays or a single array
            ts_list += item["sensor"] if isinstance(item["sensor"], list) else [item["sensor"]]
            formatted_prompts.append(item["prompt"])

        inputs = processor(
            text=formatted_prompts,
            timeseries=ts_list,
            padding=True,
            return_tensors="pt",
        )
        label = torch.tensor([item["label"] for item in batch], dtype=torch.long)

        return {
            "samples": inputs,
            "targets": label,
        }

    collate_fn = lambda batch: ChatTScollator(processor, batch)

    train_loader = DataLoader(dataset_train, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_fn)
    val_loader = DataLoader(dataset_val, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_fn)

    # 2. Feature Extraction (Freezing the Backbone)
    model = torch.compile(model)
    @torch.no_grad()
    def extract_features(loader):
        feats = []
        labels = []
        model.eval()
        model.to(device)

        #print(f"Extracting features for {loader.dataset.split} split...")
        for batch in tqdm(loader):
            samples = {k: v.to(device) for k, v in batch["samples"].items()}
            y = batch["targets"]

            # Model might return (hidden_states, other_stuff) or just hidden_states
            outputs = model(samples)
            if isinstance(outputs, (tuple, list)):
                outputs = outputs[0]
            
            # Mean pooling over the sequence length dimension (dim 1)
            # Adjust if your model architecture uses a [CLS] token at index 0
            f = torch.mean(outputs, dim=1).cpu()

            feats.append(f)
            labels.append(y)

        return torch.cat(feats, dim=0), torch.cat(labels, dim=0)

    all_train_features, all_train_labels = extract_features(train_loader)
    all_val_features, all_val_labels = extract_features(val_loader)

    all_train_features = torch.nan_to_num(all_train_features)
    all_val_features = torch.nan_to_num(all_val_features)
    
    print(f"Train features: {all_train_features.shape}, Val features: {all_val_features.shape}")

    # 4. Evaluation Function
    @torch.no_grad()
    def eval_probe(probe_val_loader):
        probe.eval()
        all_preds = []
        all_targets = []
        
        for x, y in probe_val_loader:
            x, y = x.to(device), y.to(device)
            logits = probe(x)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())
        
        all_preds = torch.cat(all_preds).numpy()
        all_targets = torch.cat(all_targets).numpy()

        # Calculate metrics
        acc = (all_preds == all_targets).mean()
        f1 = f1_score(all_targets, all_preds, average='macro')
        balanced_acc = balanced_accuracy_score(all_targets, all_preds)

        return {
            "acc": acc,
            "f1": f1,
            "balanced_acc": balanced_acc
        }

    # 5. Training Loop
    dataset_name = os.path.basename(dataset_path.rstrip("/"))
    best_acc = 0.0

    # TODO: PROBE 5 TIMES WITH DIFFERENT SEED: [0,1,2,3,4]
    all_accs = []
    all_f1s = []

    for probe_seed in [0, 1, 2, 3, 4]:
        print('Probing......')
        set_seed(probe_seed)
        # 3. Setup Probe
        num_classes = int(torch.max(all_train_labels).item()) + 1
        feat_dim = all_train_features.shape[1]
        probe = nn.Linear(feat_dim, num_classes).to(device)

        probe_train_loader = DataLoader(
            TensorDataset(all_train_features, all_train_labels),
            batch_size=probe_batch_size, shuffle=True
        )
        probe_val_loader = DataLoader(
            TensorDataset(all_val_features, all_val_labels),
            batch_size=probe_batch_size, shuffle=False
        )

        opt = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = nn.CrossEntropyLoss()

        for epoch in tqdm(range(1, epochs + 1)):
            probe.train()
            total_loss = 0.0
            
            for x, y in probe_train_loader:
                x, y = x.to(device), y.to(device)
                opt.zero_grad(set_to_none=True)
                logits = probe(x)
                loss = criterion(logits, y)
                loss.backward()
                opt.step()
                total_loss += loss.item() * x.size(0)

            metrics = eval_probe(probe_val_loader)
            train_loss = total_loss / len(all_train_features)

            # print(f"Epoch {epoch:03d} | Loss: {train_loss:.4f} | Acc: {metrics['acc']:.4f} | F1: {metrics['f1']:.4f} | B_Acc: {metrics['balanced_acc']:.4f}")

            if metrics['acc'] > best_acc:
                best_acc = metrics['acc']
                best_f1 = metrics['f1']

        
        all_accs.append(best_acc*100)
        all_f1s.append(best_f1*100)


    # print average and std of all_accs
    avg_acc = sum(all_accs) / len(all_accs)
    std_acc = (sum((x - avg_acc) ** 2 for x in all_accs) / len(all_accs)) ** 0.5
    print(f"\n{dataset_name} Average Acc over 5 runs: {avg_acc:.4f} ± {std_acc:.4f}")
    # also print the f1
    avg_f1 = sum(all_f1s) / len(all_f1s)
    std_f1 = (sum((x - avg_f1) ** 2 for x in all_f1s) / len(all_f1s)) ** 0.5
    print(f"{dataset_name} Average F1 over 5 runs: {avg_f1:.4f} ± {std_f1:.4f}")

    # clean up cuda memory, and the features..
    del model
    del probe
    del all_train_features
    del all_val_features
    torch.cuda.empty_cache()
    

# LP

In [ ]:
run_chatts_linear_probe('data/preprocessed_data/CLIP_downstream/uci_har',model,processor,)

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, I have time series 1: <ts><ts/>, I have time series 2: <ts><ts/>, I have time series 3: <ts><ts/>, I have time series 4: <ts><ts/>, I have time series 5: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, I have time series 1: <ts><ts/>, I have time series 2: <ts><ts/>, I have time series 3: <ts><ts/>, I have time series 4: <ts><ts/>, I have time series 5: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant



  0%|          | 0/116 [00:00<?, ?it/s]

100%|██████████| 50/50 [02:40<00:00,  3.21s/it]


Train features: torch.Size([1847, 4096]), Val features: torch.Size([793, 4096])
Probing......


100%|██████████| 50/50 [00:00<00:00, 63.54it/s]


...

Probing......


100%|██████████| 50/50 [00:00<00:00, 64.43it/s]


Probing......


100%|██████████| 50/50 [00:01<00:00, 26.99it/s]



uci_har Average Acc over 5 runs: 56.2421 ± 0.0000
uci_har Average F1 over 5 runs: 49.2489 ± 0.0000


In [ ]:
run_chatts_linear_probe('data/preprocessed_data/CLIP_downstream/wesad',model,processor,batch_size=2)

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, I have time series 1: <ts><ts/>, I have time series 2: <ts><ts/>, I have time series 3: <ts><ts/>, I have time series 4: <ts><ts/>, I have time series 5: <ts><ts/>, I have time series 6: <ts><ts/>, I have time series 7: <ts><ts/>, I have time series 8: <ts><ts/>, I have time series 9: <ts><ts/>, I have time series 10: <ts><ts/>, I have time series 11: <ts><ts/>, I have time series 12: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, I have time series 1: <ts><ts/>, I have time series 2: <ts><ts/>, I have time series 3: <ts><ts/>, I have time series 4: <ts><ts/>, I have time series 5: <ts><ts/>, I have time series 6: <ts><ts/>, I have time series 7: <ts><ts/>, I have time series 8: <ts><ts/>, I have time series 9: <ts><ts/

  0%|          | 0/441 [00:00<?, ?it/s]

100%|██████████| 112/112 [09:34<00:00,  5.13s/it]


Train features: torch.Size([882, 4096]), Val features: torch.Size([223, 4096])
Probing......


100%|██████████| 50/50 [00:00<00:00, 110.73it/s]


...

Probing......


100%|██████████| 50/50 [00:00<00:00, 149.34it/s]


Probing......


100%|██████████| 50/50 [00:00<00:00, 149.93it/s]



wesad Average Acc over 5 runs: 77.0404 ± 0.1794
wesad Average F1 over 5 runs: 56.5214 ± 0.1283


In [ ]:
run_chatts_linear_probe('data/preprocessed_data/CLIP_downstream/PPG_CVA',model,processor,)

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant



  0%|          | 0/33 [00:00<?, ?it/s]

100%|██████████| 9/9 [00:06<00:00,  1.46it/s]


Train features: torch.Size([525, 4096]), Val features: torch.Size([132, 4096])
Probing......


100%|██████████| 50/50 [00:00<00:00, 212.93it/s]


...

Probing......


100%|██████████| 50/50 [00:00<00:00, 232.18it/s]


Probing......


100%|██████████| 50/50 [00:00<00:00, 236.37it/s]



PPG_CVA Average Acc over 5 runs: 90.9091 ± 0.0000
PPG_CVA Average F1 over 5 runs: 47.6190 ± 0.0000


In [ ]:
run_chatts_linear_probe('data/preprocessed_data/CLIP_downstream/PPG_DM',model,processor,)

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant



100%|██████████| 9/9 [00:06<00:00,  1.42it/s]


Train features: torch.Size([522, 4096]), Val features: torch.Size([135, 4096])
Probing......


100%|██████████| 50/50 [00:00<00:00, 224.66it/s]


Probing......


...

Probing......


100%|██████████| 50/50 [00:00<00:00, 233.64it/s]


Probing......


100%|██████████| 50/50 [00:00<00:00, 232.51it/s]



PPG_DM Average Acc over 5 runs: 82.2222 ± 0.0000
PPG_DM Average F1 over 5 runs: 45.1220 ± 0.0000


In [ ]:
run_chatts_linear_probe('data/preprocessed_data/CLIP_downstream/PPG_HTN',model,processor,)

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant



  0%|          | 0/33 [00:00<?, ?it/s]

100%|██████████| 9/9 [00:06<00:00,  1.46it/s]


Train features: torch.Size([525, 4096]), Val features: torch.Size([132, 4096])
Probing......


100%|██████████| 50/50 [00:00<00:00, 133.40it/s]


...

Probing......


100%|██████████| 50/50 [00:01<00:00, 36.60it/s] 


Probing......


100%|██████████| 50/50 [00:00<00:00, 232.35it/s]



PPG_HTN Average Acc over 5 runs: 44.6970 ± 0.0000
PPG_HTN Average F1 over 5 runs: 25.0232 ± 0.0000


In [ ]:
run_chatts_linear_probe('data/preprocessed_data/CLIP_downstream/wisdm',model,processor,)

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, I have time series 1: <ts><ts/>, I have time series 2: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, I have time series 1: <ts><ts/>, I have time series 2: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant



  2%|▏         | 25/1400 [00:49<45:05,  1.97s/it]

100%|██████████| 350/350 [11:29<00:00,  1.97s/it]


Train features: torch.Size([22396, 4096]), Val features: torch.Size([5600, 4096])
Probing......


100%|██████████| 50/50 [00:09<00:00,  5.05it/s]


...

Probing......


100%|██████████| 50/50 [00:08<00:00,  5.75it/s]


Probing......


100%|██████████| 50/50 [00:09<00:00,  5.10it/s]



wisdm Average Acc over 5 runs: 75.6607 ± 0.0000
wisdm Average F1 over 5 runs: 75.3642 ± 0.0000


In [ ]:
run_chatts_linear_probe('data/preprocessed_data/CLIP_downstream/ptbxl',model,processor,batch_size=24)

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, I have time series 1: <ts><ts/>, I have time series 2: <ts><ts/>, I have time series 3: <ts><ts/>, I have time series 4: <ts><ts/>, I have time series 5: <ts><ts/>, I have time series 6: <ts><ts/>, I have time series 7: <ts><ts/>, I have time series 8: <ts><ts/>, I have time series 9: <ts><ts/>, I have time series 10: <ts><ts/>, I have time series 11: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, I have time series 1: <ts><ts/>, I have time series 2: <ts><ts/>, I have time series 3: <ts><ts/>, I have time series 4: <ts><ts/>, I have time series 5: <ts><ts/>, I have time series 6: <ts><ts/>, I have time series 7: <ts><ts/>, I have time series 8: <ts><ts/>, I have time series 9: <ts><ts/>, I have time series 10: <ts><ts/

  0%|          | 0/472 [00:00<?, ?it/s]W0204 15:37:29.125000 856181 torch/_dynamo/variables/tensor.py:1047] [0/0] Graph break from `Tensor.item()`, consider setting:
W0204 15:37:29.125000 856181 torch/_dynamo/variables/tensor.py:1047] [0/0]     torch._dynamo.config.capture_scalar_outputs = True
W0204 15:37:29.125000 856181 torch/_dynamo/variables/tensor.py:1047] [0/0] or:
W0204 15:37:29.125000 856181 torch/_dynamo/variables/tensor.py:1047] [0/0]     env TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS=1
W0204 15:37:29.125000 856181 torch/_dynamo/variables/tensor.py:1047] [0/0] to include these operations in the captured graph.
W0204 15:37:29.125000 856181 torch/_dynamo/variables/tensor.py:1047] [0/0] 
W0204 15:37:29.125000 856181 torch/_dynamo/variables/tensor.py:1047] [0/0] Graph break: from user code at:
W0204 15:37:29.125000 856181 torch/_dynamo/variables/tensor.py:1047] [0/0]   File "/scratch/leo/workspace/HealthSensorSLM-Bench/model_factory/chatts.py", line 40, in forward
W0204 15:37:29.125000 

Train features: torch.Size([11320, 4096]), Val features: torch.Size([1650, 4096])
Probing......


100%|██████████| 50/50 [00:04<00:00, 11.17it/s]


Probing......


...

Probing......


100%|██████████| 50/50 [00:03<00:00, 13.49it/s]


Probing......


100%|██████████| 50/50 [00:04<00:00, 11.77it/s]



ptbxl Average Acc over 5 runs: 67.9273 ± 0.2078
ptbxl Average F1 over 5 runs: 43.6371 ± 0.3527


In [ ]:
run_chatts_linear_probe('data/preprocessed_data/CLIP_downstream/sleepEDF',model,processor,batch_size=32)

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, I have time series 1: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, I have time series 1: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant



100%|██████████| 273/273 [46:01<00:00, 10.12s/it]


Train features: torch.Size([33599, 4096]), Val features: torch.Size([8709, 4096])
Probing......


  2%|▏         | 1/50 [00:00<00:10,  4.81it/s]

Epoch 001 | Loss: 1.3543 | Acc: 0.5524 | F1: 0.2962 | B_Acc: 0.3468
Epoch 002 | Loss: 1.1062 | Acc: 0.6243 | F1: 0.4302 | B_Acc: 0.4491


...

 98%|█████████▊| 49/50 [00:12<00:00,  4.48it/s]

Epoch 048 | Loss: 0.5295 | Acc: 0.8011 | F1: 0.6973 | B_Acc: 0.6955
Epoch 049 | Loss: 0.5281 | Acc: 0.7983 | F1: 0.6924 | B_Acc: 0.6881


100%|██████████| 50/50 [00:12<00:00,  3.96it/s]


Epoch 050 | Loss: 0.5262 | Acc: 0.8014 | F1: 0.6952 | B_Acc: 0.6938

Training Complete. Best Val Acc: 0.8053
Saved to: /scratch/leo/workspace/linear_probe_ckpt/sleepEDF/best_probe.pt


{'best_metrics': {'acc': np.float64(0.8052589275462165),
  'f1': 0.6969664095538219,
  'balanced_acc': 0.6948936227475995},
 'best_path': '/scratch/leo/workspace/linear_probe_ckpt/sleepEDF/best_probe.pt',
 'feat_dim': 4096,
 'num_classes': 5}

In [ ]:
run_chatts_linear_probe('data/preprocessed_data/CLIP_downstream/AsphaltObstacles',model,processor,)

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant



  0%|          | 0/25 [00:00<?, ?it/s]

100%|██████████| 25/25 [00:26<00:00,  1.06s/it]


Train features: torch.Size([390, 4096]), Val features: torch.Size([391, 4096])
Probing......


100%|██████████| 50/50 [00:00<00:00, 207.29it/s]


...

Probing......


100%|██████████| 50/50 [00:00<00:00, 214.00it/s]


Probing......


100%|██████████| 50/50 [00:00<00:00, 212.46it/s]



AsphaltObstacles Average Acc over 5 runs: 77.7494 ± 0.6470
AsphaltObstacles Average F1 over 5 runs: 76.6592 ± 0.3750


In [ ]:
run_chatts_linear_probe('data/preprocessed_data/CLIP_downstream/Beijing_AQI',model,processor,)

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, I have time series 1: <ts><ts/>, I have time series 2: <ts><ts/>, I have time series 3: <ts><ts/>, I have time series 4: <ts><ts/>, I have time series 5: <ts><ts/>, I have time series 6: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant

Prompt template: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
I have time series 0: <ts><ts/>, I have time series 1: <ts><ts/>, I have time series 2: <ts><ts/>, I have time series 3: <ts><ts/>, I have time series 4: <ts><ts/>, I have time series 5: <ts><ts/>, I have time series 6: <ts><ts/>, What does this suggest?<|im_end|>
<|im_start|>assistant



  0%|          | 0/73 [00:00<?, ?it/s]

100%|██████████| 19/19 [01:17<00:00,  4.08s/it]


Train features: torch.Size([1168, 4096]), Val features: torch.Size([293, 4096])
Probing......


100%|██████████| 50/50 [00:00<00:00, 98.82it/s] 


...

Probing......


100%|██████████| 50/50 [00:00<00:00, 117.55it/s]


Probing......


100%|██████████| 50/50 [00:00<00:00, 114.97it/s]



Beijing_AQI Average Acc over 5 runs: 61.2287 ± 1.8392
Beijing_AQI Average F1 over 5 runs: 31.0442 ± 1.9339


# Catch22 5 fold

In [ ]:
from evaluation.statistical_feat import extract_catch22_features
from util.dataset import EvalDataset
from types import SimpleNamespace
import yaml
with open('config/dataset/clip_downstream.yaml', 'r') as f:
    dataset_config = yaml.safe_load(f)  

dataset_config = SimpleNamespace(**dataset_config)

print(dataset_config)


/scratch/leo/envs/llm_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-04 11:17:20.774790: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-04 11:17:20.811297: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-04 11:17:21.484947: I tensorflow/core/util/port.cc:153] oneDNN custom operations are 

namespace(name='clip_downstream', input_size=[0, 0], task_name='uci_har', is_normalize=None, patch_size=None, data_dir='/scratch/leo/data/preprocessed_data/CLIP_downstream/${.task_name}', fold={'fold_0': {'train': 'None', 'val': 'None'}}, task_config={'uci_har': {'num_classes': 7, 'head_type': 'classification', 'is_normalize': False, 'patch_size': 20}, 'ecg-abnormal': {'num_classes': 5, 'head_type': 'classification', 'is_normalize': False, 'patch_size': 4}, 'drive_fatigue': {'num_classes': 2, 'head_type': 'classification', 'is_normalize': False}, 'gameemo': {'num_classes': 4, 'head_type': 'classification', 'is_normalize': False}, 'wesad': {'num_classes': 3, 'head_type': 'classification', 'is_normalize': False, 'patch_size': 16}, 'PPG_CVA': {'num_classes': 2, 'head_type': 'classification', 'is_normalize': True, 'patch_size': 4}, 'PPG_CVD': {'num_classes': 3, 'head_type': 'classification', 'is_normalize': False, 'patch_size': 16}, 'PPG_DM': {'num_classes': 2, 'head_type': 'classification

In [2]:
tasks = [
    "wisdm",
    "uci_har",
    "wesad",
    "PPG_CVA",
    "PPG_DM",
    "PPG_HTN",
    "studentlife",
    "ptbxl",
    "sleepEDF",
    "AsphaltObstacles",
    "Beijing_AQI"
]

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

seed = [0,1,2,3,4]
all_result_df = []

for task in tqdm(tasks):
    print(task)
    data = np.load(f'/scratch/leo/data/preprocessed_data/CLIP_downstream/{task}/X.npy')
    labels = np.load(f'/scratch/leo/data/preprocessed_data/CLIP_downstream/{task}/Y.npy')

    import json
    split = f'/scratch/leo/data/preprocessed_data/CLIP_downstream/{task}/split_indices.json'
    with open(split, 'r') as f:
        split_indices = json.load(f)

    train_indices = split_indices['train']
    test_indices = split_indices['test']

    X_train = data[train_indices]
    X_val = data[test_indices]
    y_train = labels[train_indices]
    y_val = labels[test_indices]

    print(f"Processing task: {task} with {X_train.shape[0]} train samples and {X_val.shape[0]} val samples")

    X_train_feat = extract_catch22_features(X_train)
    X_val_feat = extract_catch22_features(X_val)

    # do 5-fold
    global_acc = []
    global_f1 = []
    for s in seed:
        clf = RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            n_jobs=-1,
            random_state=s,
        )

        clf.fit(X_train_feat, y_train)
        y_pred = clf.predict(X_val_feat)

        acc = accuracy_score(y_val, y_pred)
        f1_macro = f1_score(y_val, y_pred, average="macro")

        global_acc.append(acc)
        global_f1.append(f1_macro)
        #print(f"[{task}] Acc: {acc:.4f} | F1(macro): {f1_macro:.4f}")
    
    # calculate mean and std
    accs = np.array(global_acc)
    f1s = np.array(global_f1)
    print(f'[{task}] Overall Results:')
    print(f'Accuracy mean: {accs.mean() * 100:.2f}%')
    print(f'Accuracy std: {accs.std() * 100:.2f}%')
    print(f'Macro F1 mean: {f1s.mean() * 100:.2f}%')
    print(f'Macro F1 std: {f1s.std() * 100:.2f}%')


  0%|          | 0/11 [00:00<?, ?it/s]

wisdm
Processing task: wisdm with 22396 train samples and 5600 val samples


  9%|▉         | 1/11 [01:10<11:43, 70.32s/it]

[wisdm] Overall Results:
Accuracy mean: 77.14%
Accuracy std: 0.25%
Macro F1 mean: 76.75%
Macro F1 std: 0.27%
uci_har
Processing task: uci_har with 1847 train samples and 793 val samples


 18%|█▊        | 2/11 [01:23<05:27, 36.44s/it]

...

[sleepEDF] Overall Results:
Accuracy mean: 77.33%
Accuracy std: 0.24%
Macro F1 mean: 65.35%
Macro F1 std: 0.33%
AsphaltObstacles
Processing task: AsphaltObstacles with 390 train samples and 391 val samples


 91%|█████████ | 10/11 [35:00<05:19, 319.04s/it]

[AsphaltObstacles] Overall Results:
Accuracy mean: 81.69%
Accuracy std: 0.80%
Macro F1 mean: 81.32%
Macro F1 std: 0.79%
Beijing_AQI
Processing task: Beijing_AQI with 1168 train samples and 293 val samples


100%|██████████| 11/11 [35:10<00:00, 191.88s/it]

[Beijing_AQI] Overall Results:
Accuracy mean: 68.81%
Accuracy std: 0.51%
Macro F1 mean: 46.93%
Macro F1 std: 2.21%
